# Audit Bethel transformed-flat data through the Upstream API

This notebook compares local `bethel1Base/data/transformed-flat` CSVs against data returned by the Upstream API, not by direct database access.

API endpoints used:

- `POST /api/v1/token`
- `GET /api/v1/campaigns/{campaign_id}/stations/{station_id}/measurements/export`

Outputs are written to `bethel1Base/data/audit-api-vs-transformed-flat/`.

In [ ]:
from __future__ import annotations

import os
import sys
from getpass import getpass
from io import StringIO
from pathlib import Path

import pandas as pd
import requests

In [ ]:
# Update this if you run the notebook from somewhere other than the repo root.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "bethel1Base" / "data" / "transformed-flat").exists():
    REPO_ROOT = Path("/Users/wmobley/Documents/GitHub/upstream")

BETHEL_DIR = REPO_ROOT / "bethel1Base"
ENV_PATH = BETHEL_DIR / ".env"
DATA_DIR = BETHEL_DIR / "data" / "transformed-flat"
OUT_DIR = BETHEL_DIR / "data" / "audit-api-vs-transformed-flat"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def load_env_file(path: Path) -> dict[str, str]:
    values: dict[str, str] = {}
    if not path.exists():
        return values
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip("'\"")
    return values


env_file = load_env_file(ENV_PATH)

def parse_unit_station_map(raw: str | None) -> dict[str, int]:
    if not raw:
        return {"unit1": 23, "unit2": 20, "unit3": 21, "unit4": 22}
    result: dict[str, int] = {}
    for item in raw.split(","):
        unit, station_id = item.split(":", 1)
        result[unit.strip()] = int(station_id.strip())
    return result


CAMPAIGN_ID = int(os.getenv("UPSTREAM_CAMPAIGN_ID") or env_file.get("UPSTREAM_CAMPAIGN_ID") or "6")
UNIT_STATION_MAP = parse_unit_station_map(os.getenv("UPSTREAM_UNIT_STATION_MAP") or env_file.get("UPSTREAM_UNIT_STATION_MAP"))

print("Environment file:", ENV_PATH, "exists=", ENV_PATH.exists())
print("Local data:", DATA_DIR)
print("Audit output:", OUT_DIR)

The notebook reads `bethel1Base/.env` using the variable names shown in `.env.example`.

For API login it uses, in order:

It authenticates using `UpstreamClient`, matching `upload_to_upstream.py`. The uploader expects `UPSTREAM_USERNAME` and `UPSTREAM_PASSWORD`; this notebook also falls back to `TAPIS_USERNAME` and `TAPIS_PASSWORD` because those are the names present in `bethel1Base/.env.example`.

In [ ]:
UPSTREAM_SDK_DIR = REPO_ROOT / "upstream-sdk"
UPSTREAM_API_CLIENT_DIR = REPO_ROOT / "upstream-python-api-client"
for path in [UPSTREAM_API_CLIENT_DIR, UPSTREAM_SDK_DIR]:
    path_str = str(path)
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

from upstream.client import UpstreamClient


BASE_URL = (
    os.getenv("UPSTREAM_BASE_URL")
    or env_file.get("UPSTREAM_BASE_URL")
    or input("UPSTREAM_BASE_URL [https://upstreamapi.pods.portals.tapis.io]: ").strip()
    or "https://upstreamapi.pods.portals.tapis.io"
).rstrip("/")
USERNAME = (
    os.getenv("UPSTREAM_USERNAME")
    or env_file.get("UPSTREAM_USERNAME")
    or os.getenv("TAPIS_USERNAME")
    or env_file.get("TAPIS_USERNAME")
    or input("TAPIS_USERNAME / UPSTREAM_USERNAME: ").strip()
)
PASSWORD = (
    os.getenv("UPSTREAM_PASSWORD")
    or env_file.get("UPSTREAM_PASSWORD")
    or os.getenv("TAPIS_PASSWORD")
    or env_file.get("TAPIS_PASSWORD")
    or getpass("TAPIS_PASSWORD / UPSTREAM_PASSWORD: ")
)

if not USERNAME:
    raise ValueError("Missing TAPIS_USERNAME or UPSTREAM_USERNAME")
if not PASSWORD:
    raise ValueError("Missing TAPIS_PASSWORD or UPSTREAM_PASSWORD")

client = UpstreamClient(username=USERNAME, password=PASSWORD, base_url=BASE_URL)
client.authenticate()

session = requests.Session()
headers = client.auth_manager.get_headers(include_tapis_token=True)
headers.pop("Content-Type", None)
session.headers.update(headers)

print("Authenticated as", client.auth_manager.username or USERNAME, "role=", client.auth_manager.role)
print("Campaign:", CAMPAIGN_ID)
print("Station map:", UNIT_STATION_MAP)

In [ ]:
def normalize_time_series(values: pd.Series) -> pd.Series:
    """Match Postgres/API timestamp precision for comparison."""
    return pd.to_datetime(values, utc=True).dt.round("us").dt.tz_convert(None)


def load_local_unit(unit: str, station_id: int) -> pd.DataFrame:
    sensors_path = DATA_DIR / f"{unit}_sensors.csv"
    measurements_path = DATA_DIR / f"{unit}_measurements.csv"

    sensors = pd.read_csv(sensors_path, keep_default_na=False)
    aliases = sensors["alias"].astype(str).tolist()

    measurements = pd.read_csv(measurements_path, keep_default_na=False)
    measurements["collectiontime_norm"] = normalize_time_series(measurements["collectiontime"])

    long = measurements.melt(
        id_vars=["collectiontime", "collectiontime_norm", "Lat_deg", "Lon_deg"],
        value_vars=[alias for alias in aliases if alias in measurements.columns],
        var_name="alias",
        value_name="local_value",
    )
    long = long[long["local_value"].astype(str).str.strip() != ""].copy()
    long["local_value"] = pd.to_numeric(long["local_value"], errors="coerce")
    long["unit"] = unit
    long["stationid"] = station_id
    return long[
        ["unit", "stationid", "alias", "collectiontime", "collectiontime_norm", "Lat_deg", "Lon_deg", "local_value"]
    ]


def load_api_station_export(unit: str, station_id: int) -> pd.DataFrame:
    url = f"{BASE_URL}/api/v1/campaigns/{CAMPAIGN_ID}/stations/{station_id}/measurements/export"
    response = session.get(url, timeout=300)
    response.raise_for_status()

    export_path = OUT_DIR / f"{unit}_station_{station_id}_api_export.csv"
    export_path.write_text(response.text, encoding="utf-8")

    if response.text.lstrip().startswith("# Error"):
        raise RuntimeError(response.text[:1000])

    wide = pd.read_csv(StringIO(response.text), keep_default_na=False)
    if wide.empty:
        return pd.DataFrame(columns=["unit", "stationid", "alias", "collectiontime", "collectiontime_norm", "api_value"])

    sensor_cols = [col for col in wide.columns if col not in {"collectiontime", "Lat_deg", "Lon_deg"}]
    wide["collectiontime_norm"] = normalize_time_series(wide["collectiontime"])
    long = wide.melt(
        id_vars=["collectiontime", "collectiontime_norm", "Lat_deg", "Lon_deg"],
        value_vars=sensor_cols,
        var_name="alias",
        value_name="api_value",
    )
    long = long[long["api_value"].astype(str).str.strip() != ""].copy()
    long["api_value"] = pd.to_numeric(long["api_value"], errors="coerce")
    long["unit"] = unit
    long["stationid"] = station_id
    return long[["unit", "stationid", "alias", "collectiontime", "collectiontime_norm", "api_value"]]

In [ ]:
local_df = pd.concat(
    [load_local_unit(unit, station_id) for unit, station_id in UNIT_STATION_MAP.items()],
    ignore_index=True,
)

print(f"Loaded {len(local_df):,} local measurement values from {DATA_DIR}")
local_counts = (
    local_df.groupby(["unit", "stationid", "alias"], as_index=False)
    .size()
    .rename(columns={"size": "local_count"})
)
local_counts

In [ ]:
api_df = pd.concat(
    [load_api_station_export(unit, station_id) for unit, station_id in UNIT_STATION_MAP.items()],
    ignore_index=True,
)

print(f"Loaded {len(api_df):,} API-exported measurement values")
api_counts = (
    api_df.groupby(["unit", "stationid", "alias"], as_index=False)
    .size()
    .rename(columns={"size": "api_count"})
    .sort_values(["unit", "alias"])
)
api_counts

In [ ]:
key_cols = ["stationid", "alias", "collectiontime_norm"]
local_keys = local_df[key_cols + ["unit", "collectiontime", "Lat_deg", "Lon_deg", "local_value"]]
api_keys = api_df[key_cols + ["collectiontime", "api_value"]].rename(
    columns={"collectiontime": "api_collectiontime"}
)

comparison = local_keys.merge(api_keys, on=key_cols, how="left", indicator=True)
missing = comparison[comparison["_merge"] == "left_only"].drop(columns=["_merge"]).copy()
present = comparison[comparison["_merge"] == "both"].drop(columns=["_merge"]).copy()

api_extra = api_keys.merge(local_keys[key_cols], on=key_cols, how="left", indicator=True)
api_extra = api_extra[api_extra["_merge"] == "left_only"].drop(columns=["_merge"]).copy()
station_to_unit = {station_id: unit for unit, station_id in UNIT_STATION_MAP.items()}
api_extra["unit"] = api_extra["stationid"].map(station_to_unit)

summary = (
    local_counts.merge(api_counts, on=["unit", "stationid", "alias"], how="outer")
    .merge(
        missing.groupby(["unit", "stationid", "alias"], as_index=False)
        .size()
        .rename(columns={"size": "missing_count"}),
        on=["unit", "stationid", "alias"],
        how="left",
    )
    .merge(
        api_extra.groupby(["unit", "stationid", "alias"], as_index=False)
        .size()
        .rename(columns={"size": "api_extra_count"}),
        on=["unit", "stationid", "alias"],
        how="left",
    )
    .fillna({"local_count": 0, "api_count": 0, "missing_count": 0, "api_extra_count": 0})
)

count_cols = ["local_count", "api_count", "missing_count", "api_extra_count"]
summary[count_cols] = summary[count_cols].astype(int)
summary = summary.sort_values(["unit", "alias"])

summary_path = OUT_DIR / "summary_by_unit_alias.csv"
missing_path = OUT_DIR / "missing_local_rows.csv"
extra_path = OUT_DIR / "api_extra_rows.csv"

summary.to_csv(summary_path, index=False)
missing.to_csv(missing_path, index=False)
api_extra.to_csv(extra_path, index=False)

print(f"Wrote {summary_path}")
print(f"Wrote {missing_path}")
print(f"Wrote {extra_path}")
summary

In [ ]:
if missing.empty:
    print("No local transformed-flat rows are missing from the API export for the configured stations.")
else:
    print(f"{len(missing):,} local transformed-flat rows are missing from the API export.")

missing.sort_values(["unit", "alias", "collectiontime_norm"]).head(100)

In [ ]:
value_check = present.copy()
value_check["abs_delta"] = (value_check["local_value"] - value_check["api_value"]).abs()
value_mismatches = value_check[value_check["abs_delta"] > 1e-9].copy()

value_mismatch_path = OUT_DIR / "value_mismatches.csv"
value_mismatches.to_csv(value_mismatch_path, index=False)

print(f"Wrote {value_mismatch_path}")
print(f"Value mismatches: {len(value_mismatches):,}")
value_mismatches.sort_values(["unit", "alias", "collectiontime_norm"]).head(100)